<a href="https://colab.research.google.com/github/francischisom/BackPropagation-Algorithm/blob/main/SVM_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# SVM-BERT Hybrid (frozen embeddings + SVM)

**Expected Outputs:** `metrics_svm_bert.csv`, `gap_svm_bert.csv`,
`probs_svm_bert_val.csv`, `probs_svm_bert_test.csv`.

## Setup

In [1]:
# Install PyTorch, Hugging Face Transformers, and numerical/ML dependencies
!pip install -q torch transformers numpy pandas scikit-learn
import sys; sys.path.append('.')   # Ensure .py modules are discoverable in sys.path
import eval_utils as ev

# Load preprocessed splits: train for fine-tuning, val for early stopping, test for final evaluation
train_df, val_df, test_df, full_df = ev.load_splits(
    'train.csv', 'val.csv', 'test.csv', 'unified_full.csv')

# Assign stable row identifiers to validation and test sets
# Critical for ensemble stacking: allows probability alignment across heterogeneous models
val_df  = ev.add_uid(val_df)
test_df = ev.add_uid(test_df)

# Confirm data shapes and compute device (CPU/GPU) for reproducibility and performance tuning
print('Train', train_df.shape, '| Val', val_df.shape, '| Test', test_df.shape)
print('Device:', ev.get_device())

Train (2138, 9) | Val (456, 10) | Test (450, 10)
Device: cuda


## Train + standard evaluation
Embeddings are cached so val/test are encoded once each.

In [2]:
# Train SVM on BERT embeddings:
import svm_bert as sb

# Cache stores BERT encodings to avoid redundant forward passes across train/test
cache = {}
std, clf = sb.train_and_eval(train_df, test_df, cache=cache)
print('Standard test:', {k: round(v,3) for k,v in std.items()})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Standard test: {'accuracy': 0.978, 'precision': 0.978, 'recall': 0.985, 'f1': 0.981, 'f1_macro': 0.977, 'roc_auc': np.float64(0.993)}


## Cross-domain

In [3]:
# Cross-domain evaluation: train on Formal (DailyMed), test on Informal (CADEC)
# Validates SVM+BERT generalization across structured labels vs. patient narratives
cross = sb.run_cross_domain(full_df, 'Formal', 'Informal')
print('Cross-domain:', {k: round(v,3) for k,v in cross.items()})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-domain: {'accuracy': 0.989, 'precision': 0.99, 'recall': 0.998, 'f1': 0.994, 'f1_macro': 0.964, 'roc_auc': np.float64(0.987)}


## Export results + probabilities

In [4]:
import pandas as pd
MODEL = 'svm_bert'

# Export standard test metrics (F1, precision, recall, ROC-AUC) for final report
pd.DataFrame([{'model':'SVM-BERT', **std}]).to_csv(
    f'metrics_{MODEL}.csv', index=False)

# Quantify domain generalization gap: (standard F1) - (cross-domain F1)
# Negative values indicate performance drop on unseen domain
pd.DataFrame([ev.degradation_row('SVM-BERT', std['f1'],
             cross['f1'])]).to_csv(f'gap_{MODEL}.csv', index=False)

# Reuse cached BERT embeddings from training (efficiency); embed val set for meta-learner
# test_proba extracted directly from cache; val_proba requires fresh encoding
test_proba = clf.predict_proba(cache['test'])[:, 1]
val_proba  = sb.predict_proba_on_df(clf, val_df)

# Save class probabilities keyed by row UID for ensemble stacking
ev.save_probabilities(val_df,  val_proba,  MODEL, f'probs_{MODEL}_val.csv')
ev.save_probabilities(test_df, test_proba, MODEL, f'probs_{MODEL}_test.csv')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved 456 probabilities -> probs_svm_bert_val.csv
Saved 450 probabilities -> probs_svm_bert_test.csv


,uid,label,svm_bert
0,73fba02e8dd1418f,0,0.013970
1,6dae2bf4cbcaf6e1,0,0.143110
2,81afbccca2b34d46,0,0.003776
3,946daef461e9f47e,0,0.008182
4,0ea1f852bc98f7ba,0,0.106595
...,...,...,...
445,4a564bc52f16cbad,1,0.999983
446,dddf223894e4c088,1,0.992084
447,99eb8929dc425561,1,1.000000
448,2389bb2dc9ae62d6,1,0.999984
